In [7]:
start_date = "2024-01-01"
tkrs = ["NVDA", "AAPL", "AMZN", "MSFT", "TSLA"]

In [8]:
import pandas as pd
import numpy as np
import yfinance as yf

tkr_data = yf.Tickers(tkrs).history(start=start_date)
tkr_d = {
    idx: gp.xs(idx, level=0, axis=1)
    for idx, gp in tkr_data.swaplevel(axis=1).groupby(level=0, axis=1)
}

for tkr, df in tkr_d.items():
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"]) - np.log(df["Close"].shift(1))

[*********************100%***********************]  5 of 5 completed
/tmp/ipykernel_194086/2633914690.py:8: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  for idx, gp in tkr_data.swaplevel(axis=1).groupby(level=0, axis=1)


In [9]:
from pandas_datareader.famafrench import get_available_datasets
import pandas_datareader.data as web

ds = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench")

/tmp/ipykernel_194086/669063850.py:4: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench")


In [10]:
factors = ds[0].loc[start_date:]
factors.index = factors.index.tz_localize("UTC")
factors

,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
2023-01-03 00:00:00+00:00,-0.48,0.05,-0.12,0.25,0.51,0.017
2023-01-04 00:00:00+00:00,0.81,0.56,0.05,-0.78,-0.02,0.017
2023-01-05 00:00:00+00:00,-1.14,0.14,1.22,0.78,0.80,0.017
2023-01-06 00:00:00+00:00,2.21,-0.01,0.05,0.85,0.14,0.017
2023-01-09 00:00:00+00:00,0.04,0.34,-1.24,-0.69,-1.01,0.017
...,...,...,...,...,...,...
2024-08-26 00:00:00+00:00,-0.34,0.33,0.16,0.13,-0.06,0.022
2024-08-27 00:00:00+00:00,0.06,-0.90,0.02,0.27,0.23,0.022
2024-08-28 00:00:00+00:00,-0.67,-0.22,1.14,0.55,-0.16,0.022


In [11]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

results = {}
for tkr, df in tkr_d.items():
    reg_df = df.join(factors)
    reg_df = reg_df.rename({"Mkt-RF": "MktRF"}, axis="columns")
    mod = smf.ols(formula="returns ~ MktRF + SMB + HML + RMW + CMA", data=reg_df)
    res = mod.fit()
    results[tkr] = res.params

result_df = pd.DataFrame(results)
result_df["tot"] = result_df.sum(axis=1)
result_df

,AAPL,AMZN,MSFT,NVDA,TSLA,tot
Intercept,0.000452,0.000180,-0.000055,0.002727,0.000349,0.003653
MktRF,0.010355,0.012029,0.010138,0.022044,0.018810,0.073376
SMB,0.001437,-0.002630,-0.002466,-0.005355,0.001065,-0.007950
HML,-0.005202,-0.001971,-0.004130,-0.010119,-0.003548,-0.024968
RMW,0.006486,-0.003468,0.003008,0.006956,-0.004810,0.008171
CMA,-0.000256,-0.012933,-0.007785,-0.008362,-0.006592,-0.035929
